In [1]:
import torch

data = torch.tensor(
    [
        [2,3,3],
        [5,6,4],
        [7,8,5]
    ]
)
data

tensor([[2, 3, 3],
        [5, 6, 4],
        [7, 8, 5]])

In [2]:
X = data[:,0:2].float()
y = data[:,2].float()

In [3]:
X

tensor([[2., 3.],
        [5., 6.],
        [7., 8.]])

In [4]:
y

tensor([3., 4., 5.])

# Weight Initialization

In [5]:
# w = 1, b = 0

def initialize_parameters():
  parameters = {}

  parameters['W1'] = torch.ones( (2,2) ) * 0.1
  parameters['b1'] = torch.zeros( (2,1) )


  parameters['W2'] = torch.ones( (2,1) ) * 0.1

  parameters['b2'] = torch.zeros( (1,1) )

  return parameters



In [6]:
initialize_parameters()

{'W1': tensor([[0.1000, 0.1000],
         [0.1000, 0.1000]]),
 'b1': tensor([[0.],
         [0.]]),
 'W2': tensor([[0.1000],
         [0.1000]]),
 'b2': tensor([[0.]])}

# Better way

In [7]:
def initialize_parameters(layer_dims):
  torch.manual_seed(3)
  parameters = {}
  L = len(layer_dims)

  for l in range(1,L):
    parameters["W" + str(l)] = torch.ones( (layer_dims[l-1], layer_dims[l]))*0.1
    parameters["b" + str(l)] = torch.zeros( (layer_dims[l], 1))

  return parameters




In [8]:
initialize_parameters( [2,2,1] )

{'W1': tensor([[0.1000, 0.1000],
         [0.1000, 0.1000]]),
 'b1': tensor([[0.],
         [0.]]),
 'W2': tensor([[0.1000],
         [0.1000]]),
 'b2': tensor([[0.]])}

# Forward Propagation

In [9]:
def linear_forward( A_prev, W, b ) :
  Z = torch.matmul( W.T , A_prev ) + b
  return Z


In [10]:
def L_layer_forward(X,parameters):
  A = X
  L = len(parameters)//2

  for l in range(1,L+1):
    A_prev = A

    wl = parameters['W' + str(l)]
    bl = parameters['b' + str(l)]
    A = linear_forward( A_prev, wl, bl )

  return A, A_prev





In [11]:
X

tensor([[2., 3.],
        [5., 6.],
        [7., 8.]])

In [12]:
params = initialize_parameters([2,2,1])
X_sample = X[0].reshape(2,1)
y_hat, A1 = L_layer_forward(X_sample,params)


In [13]:
y_hat

tensor([[0.1000]])

In [14]:
A1

tensor([[0.5000],
        [0.5000]])

In [15]:
(y-y_hat) ** 2

tensor([[ 8.4100, 15.2100, 24.0100]])

In [16]:
params

{'W1': tensor([[0.1000, 0.1000],
         [0.1000, 0.1000]]),
 'b1': tensor([[0.],
         [0.]]),
 'W2': tensor([[0.1000],
         [0.1000]]),
 'b2': tensor([[0.]])}

In [17]:
params['W2']

tensor([[0.1000],
        [0.1000]])

In [18]:
params['W2'][0,0]

tensor(0.1000)

In [19]:
params['W2'][1,0]

tensor(0.1000)

In [20]:
A1

tensor([[0.5000],
        [0.5000]])

In [21]:
def update_parameters(parameters, y, y_hat, A1, X, lr=0.001):

    err_signal = 2 * (y - y_hat)

     #  Save OLD W2 values before updating
    W2_00_old = parameters['W2'][0, 0].item()
    W2_10_old = parameters['W2'][1, 0].item()

    # Update Layer 2




    parameters['W2'][0, 0] += lr * err_signal * A1[0, 0]
    parameters['W2'][1, 0] += lr * err_signal * A1[1, 0]
    parameters['b2'][0, 0] += lr * err_signal #


    parameters['W1'][0, 0] += lr * err_signal * parameters['W2'][0, 0] * X[0, 0]
    parameters['W1'][0, 1] += lr * err_signal * parameters['W2'][0, 0] * X[1, 0]
    parameters['b1'][0, 0] += lr * err_signal * parameters['W2'][0, 0]

    parameters['W1'][1, 0] += lr * err_signal * parameters['W2'][1, 0] * X[0, 0]
    parameters['W1'][1, 1] += lr * err_signal * parameters['W2'][1, 0] * X[1, 0]
    parameters['b1'][1, 0] += lr * err_signal * parameters['W2'][1, 0]

    return parameters

In [22]:
# using loop



params = initialize_parameters([2, 2, 1])
epochs = 5

for i in range(epochs):
    epoch_loss = 0
    for j in range(X.shape[0]):
        # Prepare sample
        X_sample = X[j].reshape(2, 1)
        y_sample = y[j]

        # Forward
        y_hat, A1 = L_layer_forward(X_sample, params)
        y_hat_scalar = y_hat[0, 0]

        # Update
        params = update_parameters(params, y_sample, y_hat_scalar, A1, X_sample)

        # Loss calculation
        loss = (y_sample - y_hat_scalar) ** 2
        epoch_loss += loss.item()

    print(f"Epoch - {i+1} Loss - {epoch_loss / X.shape[0]}")

Epoch - 1 Loss - 14.67866039276123
Epoch - 2 Loss - 13.701451301574707
Epoch - 3 Loss - 12.494118690490723
Epoch - 4 Loss - 11.040447076161703
Epoch - 5 Loss - 9.366330782572428
